# 3DMapping — Free Google Colab Reconstruction

This notebook is the **$0 GPU path** for the 3DMapping project. It takes a drone video, samples frames, runs COLMAP for camera poses, trains a Gaussian Splat with Nerfstudio/Splatfacto, and exports a `.ply` file for the project's browser viewer.

The free Colab runtime is temporary. Download the final `.ply` before the session ends.

In [ ]:
# Check the free runtime GPU.
!nvidia-smi
import sys, torch
print('Python:', sys.version)
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('No CUDA GPU is attached. In Colab choose Runtime > Change runtime type > GPU.')
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# Install a known-compatible free-Colab stack.
# Nerfstudio 1.1.5 uses av and gsplat; pinning the versions avoids current pip/build skew.
!python -m pip install -q 'pip<24.1'
!python -m pip install -q 'torch==2.3.1' 'torchvision==0.18.1' --index-url https://download.pytorch.org/whl/cu121
!python -m pip install -q 'av==12.3.0'
!python -m pip install -q 'nerfstudio==1.1.5'
!apt-get update -qq
!apt-get install -y -qq ffmpeg colmap

import torch
print('PyTorch after install:', torch.__version__)
print('CUDA after install:', torch.cuda.is_available())
!colmap -h | head -n 5
!ns-train --help | head -n 8

In [ ]:
# Upload a drone video from your Mac.
from google.colab import files
from pathlib import Path

uploaded = files.upload()
if not uploaded:
    raise RuntimeError('No video selected.')
video_name = next(iter(uploaded))
video_path = Path('/content') / video_name
print('Video:', video_path)
!ffprobe -v error -show_entries format=duration -show_entries stream=width,height,r_frame_rate,codec_name -of default=noprint_wrappers=1 "{video_path}"

In [ ]:
# Extract a manageable sequential image set.
# Start at 3 FPS and cap the run to 80 frames for the free T4.
import shutil, subprocess
from pathlib import Path

root = Path('/content/3dmapping')
images = root / 'images'
if root.exists(): shutil.rmtree(root)
images.mkdir(parents=True)

fps = 3
max_frames = 80
max_width = 1280
subprocess.run([
    'ffmpeg','-y','-i',str(video_path),
    '-vf',f'fps={fps},scale={max_width}:-2:force_original_aspect_ratio=decrease',
    '-frames:v',str(max_frames),
    '-q:v','2', str(images/'frame_%06d.jpg')
], check=True)

count = len(list(images.glob('*.jpg')))
print('Extracted frames:', count)
if count < 20:
    raise RuntimeError('Too few frames were extracted for a useful reconstruction.')

In [ ]:
# COLMAP: feature extraction + sequential matching + incremental SfM.
# The Ubuntu/Colab COLMAP package may be CPU-only, so explicitly use CPU here.
# Gaussian Splat training below is the GPU-heavy stage.
db = root / 'database.db'
sparse = root / 'sparse'
sparse.mkdir(exist_ok=True)

!colmap feature_extractor --database_path "{db}" --image_path "{images}" --ImageReader.single_camera 1 --FeatureExtraction.use_gpu 0
!colmap sequential_matcher --database_path "{db}" --SequentialMatching.overlap 10 --FeatureMatching.use_gpu 0
!colmap mapper --database_path "{db}" --image_path "{images}" --output_path "{sparse}"

models = sorted([p for p in sparse.iterdir() if p.is_dir() and p.name.isdigit()], key=lambda p:int(p.name))
if not models:
    raise RuntimeError('COLMAP did not produce a sparse model. Try footage with more overlap, visible texture, and slower camera motion.')
model = models[0]
print('COLMAP model:', model)
print('Model files:', [p.name for p in model.iterdir()])

In [ ]:
# Prepare a Nerfstudio dataset from the COLMAP result.
processed = root / 'ns_data'
if processed.exists(): shutil.rmtree(processed)

!ns-process-data images --data "{images}" --output-dir "{processed}" --colmap-model-path "{model}" --skip-image-processing

if not (processed / 'transforms.json').exists():
    raise RuntimeError('Nerfstudio preprocessing did not create transforms.json.')
print('Prepared dataset:', processed)

In [ ]:
# Train Gaussian Splatting on the T4.
# 15000 iterations is a deliberately small first run for free Colab.
output_dir = root / 'nerfstudio_output'
iterations = 15000

!ns-train splatfacto --data "{processed}" --output-dir "{output_dir}" --max-num-iterations {iterations} --viewer.quit-on-train-completion True

configs = sorted(output_dir.glob('*/**/config.yml'))
if not configs:
    raise RuntimeError('Splatfacto training did not produce a config.yml.')
config = configs[-1]
print('Training config:', config)

In [ ]:
# Export the trained Gaussian Splat as PLY for the 3DMapping viewer.
export_dir = root / 'exported_ply'
export_dir.mkdir(exist_ok=True)
!ns-export gaussian-splat --load-config "{config}" --output-dir "{export_dir}"

ply_files = sorted(export_dir.rglob('*.ply'))
if not ply_files:
    raise RuntimeError('No PLY export was produced.')
ply = ply_files[-1]
print('FINAL PLY:', ply)
print('Size MB:', round(ply.stat().st_size / 1024 / 1024, 2))

In [ ]:
# Download the result to your Mac before the free runtime ends.
from google.colab import files
files.download(str(ply))

## Result

A successful export proves the real COLMAP → Gaussian Splat pipeline worked. It does not mean every possible drone video will reconstruct successfully. COLMAP must first register enough views into one consistent camera solution.

If COLMAP fails, improve overlap, texture, viewpoint diversity, and motion smoothness rather than forcing the splat stage.